# Case study: a supply chain that amplifies demand {#sec-case-supply}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/c13-supply-chain.ipynb)

A paint maker in Malaysia sells through regional wholesalers to hardware shops. Sales in the shops rose by a tenth after a housing programme started, a modest and lasting change. Yet within three months the factory was running overtime on orders half as large again as usual, and a few months later it was idle while its warehouse overflowed. Each manager in the chain says they ordered sensibly. This chapter asks why a small change at the till became a large swing at the factory, and what the chain can do about it.

The swing has a name, the **bullwhip effect**: variation in orders grows at each step up a supply chain, away from the final customer. It is the oscillation of chapter 6 repeated stage after stage, each stage managing its own stock through a delay and passing a distorted signal to the next. The case is a classic of system dynamics: Forrester modelled it in a production and distribution chain [@forrester1961], and Sterman's experiments with the beer distribution game showed that people playing the roles of the stages produce it reliably [@sterman1989]. Our version is a compact model of our own, built from the molecules of chapter 6.

**Who it is for.** Analysts who will model any chain of organisations that order from each other: retail and manufacturing, food, medicines, spare parts, blood and vaccines. **Who can skip.** If you have built a multi-stage stock management model before, read sections 5 and 6 and the brief. **Prerequisites.** Chapter 2, From story to diagram (reference modes, boundaries and dynamic hypotheses); chapter 4, Delays, pipelines and memory (material delays and smoothing); chapter 6, Why things swing (the Stock Management and Low-Visibility Pipeline Correction molecules, supply line weight); chapter 9, Testing your model; and chapter 11, From model to decision (policy runs, scenario grids and regret). **Time.** About three hours with the exercises.

The data in this chapter are synthetic, generated in the notebook and labelled as such. The firm is invented.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package,
# which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cycler import cycler

SEED = 13
rng = np.random.default_rng(SEED)

# Okabe-Ito colours with a different line style each, readable in greyscale
OKABE_ITO = ["#0072B2", "#D55E00", "#009E73", "#CC79A7",
             "#E69F00", "#56B4E9", "#000000", "#F0E442"]
STYLES = ["-", "--", ":", "-.", (0, (5, 1)), (0, (3, 1, 1, 1)),
          (0, (1, 3)), (0, (8, 3))]
plt.rcParams["axes.prop_cycle"] = (cycler(color=OKABE_ITO)
                                   + cycler(linestyle=STYLES))
plt.rcParams.update({"figure.figsize": (7, 3.6), "axes.grid": True,
                     "grid.alpha": 0.3, "lines.linewidth": 1.8})
print("loopbricks", lb.__version__)

In [ ]:
#| fig-alt: "Stock-and-flow sketch of the chain. Goods move left to right: production starts fill the Factory production line, then Factory inventory, Wholesaler in transit, Wholesaler inventory, Retailer in transit and Retailer inventory, and leave as sales. Below, orders move right to left into the Retailer, Wholesaler and Factory backlogs. Dashed arrows run from each inventory to the order decision of its stage."
from matplotlib.patches import Circle, FancyBboxPatch


def draw_sfd(ax, stocks, flows, links):
    """A small stock-and-flow diagram (the helper of chapter 6).

    stocks: {name: (x, y)}. flows: [(label, (x0, y0), (x1, y1))], drawn as
    pipes with a valve; an end outside every stock gets a cloud.
    links: [(start, end, label, curvature, label position)], dashed arrows.
    """
    w, h = 1.7, 0.8
    for name, (x, y) in stocks.items():
        ax.add_patch(FancyBboxPatch((x - w / 2, y - h / 2), w, h,
                                    boxstyle="round,pad=0.02", fc="white",
                                    ec="black", lw=1.5))
        ax.text(x, y, name, ha="center", va="center", fontsize=8)

    def inside(p):
        return any(abs(p[0] - x) <= w / 2 + 1e-9 and abs(p[1] - y) <= h / 2
                   for x, y in stocks.values())

    for label, a, b in flows:
        for p in (a, b):
            if not inside(p):
                ax.text(*p, "\u2601", ha="center", va="center",
                        fontsize=18, color="grey")
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="-|>", lw=3, color="black",
                                    shrinkA=8, shrinkB=2))
        mx, my = (a[0] + b[0]) / 2, (a[1] + b[1]) / 2
        ax.add_patch(Circle((mx, my), 0.12, fc="white", ec="black",
                            lw=1.2, zorder=3))
        ax.text(mx, my - 0.3, label, ha="center", va="top", fontsize=7)
    for a, b, label, rad, where in links:
        ax.annotate("", xy=b, xytext=a,
                    arrowprops=dict(arrowstyle="->", lw=1, ls="--",
                                    color="#0072B2", shrinkA=4, shrinkB=4,
                                    connectionstyle=f"arc3,rad={rad}"))
        ax.text(*where, label, ha="center", fontsize=7, color="#0072B2")
    ax.set_aspect("equal")
    ax.axis("off")


xs = [1.5, 4.5, 7.5, 10.5, 13.5, 16.5]   # goods row, factory to retailer
goods = ["Production\nline", "Factory\ninventory",
         "Wholesaler\nin transit", "Wholesaler\ninventory",
         "Retailer\nin transit", "Retailer\ninventory"]
backlogs = {"Factory\nbacklog": (4.5, -2.4),
            "Wholesaler\nbacklog": (10.5, -2.4),
            "Retailer\nbacklog": (16.5, -2.4)}
steps = ["production", "shipments", "arrivals", "shipments", "arrivals"]
fig, ax = plt.subplots(figsize=(12, 3.4))
draw_sfd(ax,
         stocks={**dict(zip(goods, [(x, 0) for x in xs])), **backlogs},
         flows=[("starts", (-0.6, 0), (0.65, 0))]
         + [(lab, (x0 + 0.85, 0), (x1 - 0.85, 0))
            for lab, x0, x1 in zip(steps, xs[:-1], xs[1:])]
         + [("sales", (17.35, 0), (18.8, 0)),
            ("wholesaler orders", (7.5, -2.4), (5.35, -2.4)),
            ("retailer orders", (13.5, -2.4), (11.35, -2.4)),
            ("customer orders", (18.8, -2.4), (17.35, -2.4))],
         links=[((4.5, -0.4), (0.1, -0.2), "factory decides", -0.3,
                 (2.2, -1.5)),
                ((10.5, -0.4), (6.45, -2.2), "wholesaler decides", 0.2,
                 (9.5, -1.6)),
                ((16.5, -0.4), (12.45, -2.2), "retailer decides", 0.2,
                 (15.5, -1.6))])
ax.set_xlim(-1.0, 19.3)
ax.set_ylim(-3.0, 0.7)
ax.set_title("Three stages, two directions: goods go down the chain, "
             "orders go up")
plt.show()

Figure: The chain as a stock-and-flow sketch. Goods (top row) move from the factory's production line to the retailer's shelves and out as sales; orders (bottom row) move the other way and wait in each supplier's backlog until goods are shipped. Each dashed arrow is a stage's ordering decision, which looks at that stage's inventory and at what it already has on order.

**Learning outcomes.** By the end of the chapter you can:

1. **measure** amplification in a reference mode with two ratios, and say what each hides (Try it 1);
2. **justify** a dynamic hypothesis and a model boundary for a supply chain, starting from the Stock Management rule (Try it 2);
3. **build** a three-stage chain with `Model.from_spec` from a function that writes one stage (Try it 3);
4. **verify** the chain with conservation, extreme-condition and time-step tests (Try it 4);
5. **explain** amplification through the terms of the ordering rule and the delays they ignore (Try it 5);
6. **compare** policies on amplification and cost across scenarios, and **write** a brief for the chain's managers (Try it 6 and the mini-project).

## 1. The problem and the reference mode

A **reference mode** (chapter 2) is the pattern over time that the model must explain. Our firm has no clean records, so we build a synthetic one that has the shape the managers describe: customer demand steps up by 10% in week 5 and stays there, and orders at each stage swing above and below the new level before settling, more and later at each step up the chain. The series below are drawn by hand from damped sine waves plus noise (`rng.normal` adds random noise with the chapter's seed). They are not real data, and they are deliberately not output of the model we build later, so the model has something independent to explain.

We measure amplification with the **amplification ratio**: how far a series rises above its old level at its peak, divided by the lasting change in customer demand. A ratio of 1 means a stage passes the change on as it is; a ratio of 4 means it orders four times the change at the peak.

In [ ]:
#| fig-alt: "Synthetic weekly orders over 60 weeks. Customer demand steps from about 100 to 110 cases a week in week 5. Retailer orders swing a little above it, wholesaler orders more and later, factory orders most and latest, peaking near 160 cases a week before settling at 110."
# Synthetic records, labelled as such: a stylised version of the pattern
# reported for supply chains, drawn by hand. They are not real data and
# not output of the model we build later.
weeks = np.arange(61)                                  # week 0 to 60
step = np.where(weeks >= 5, 10.0, 0.0)   # 10 cases/week from week 5, else 0


def swing(t, amp, lag, period=16, decay=12):
    """A damped swing (cases/week) starting `lag` weeks after the step."""
    s = np.clip(t - 5 - lag, 0, None)                  # weeks since start
    return amp * np.sin(2 * np.pi * s / period) * np.exp(-s / decay)


ref = pd.DataFrame(index=pd.Index(weeks, name="week"))
ref["Customer demand"] = 100 + step + rng.normal(0, 1.5, weeks.size)
for name, amp, lag in [("Retailer orders", 20, 0),
                       ("Wholesaler orders", 40, 1.5),
                       ("Factory orders", 70, 3)]:
    ref[name] = 100 + step + swing(weeks, amp, lag) \
        + rng.normal(0, 2, weeks.size)

fig, ax = plt.subplots()
ref.plot(ax=ax)
ax.set(title="Weekly orders at each stage (synthetic records)",
       xlabel="time (week)", ylabel="orders (cases/week)")
ax.legend(fontsize=8)
plt.show()

before = ref.loc[:4].mean()                            # average, weeks 0-4
after = ref.loc[40:].mean()                            # average, weeks 40-60
summary = pd.DataFrame({"before": before, "after": after,
                        "peak": ref.max(), "peak week": ref.idxmax(),
                        "low after peak": [ref.loc[w:, c].min() for c, w
                                           in ref.idxmax().items()]})
summary["amplification"] = ((summary["peak"] - summary["before"])
                            / (after - before)["Customer demand"])
print(summary.round(1).to_string())

Figure: A synthetic reference mode for the paint chain (generated in this notebook, seeded, not real data). Customer demand steps from 100 to 110 cases a week in week 5; orders at each stage overshoot, more and later the further the stage is from the customer.

The pattern has three features that a model must reproduce. **Amplification grows up the chain**: the retailer's orders peak at 2.2 times the change in demand, the wholesaler's at 4.1 and the factory's at 6.4. **Peaks come later up the chain**: weeks 8, 9 and 11. **Orders undershoot after the peak**: after its peak, the factory's orders fall to 81.6 cases a week, although customers are buying about 110. The ratio for customer demand itself (1.3) is noise: one week of 112.9 cases against a new level of 110.3, a reminder that a peak-based measure on noisy data is rough. The **time horizon** follows from the pattern: the swings die out in about 40 weeks, so the model runs for 100.

**Try it 1.**

- *Check.* From the table, how many weeks pass between the retailer's peak and the factory's?
- *Apply.* The **bullwhip ratio** is the more common measure: the standard deviation of a stage's orders divided by that of customer demand. Compute it from week 5 on.
- *Extend.* Compute the same ratio from week 40 on, when the swing has died down. What does the change tell you about using the ratio on short or long records?

<details><summary>Worked answer</summary>

*Check.* Three weeks: the retailer peaks in week 8 and the factory in week 11, with the wholesaler in between in week 9.

*Apply.* From week 5 on, the ratios are 3.4 for the retailer, 7.0 for the wholesaler and 11.7 for the factory. They are larger than the peak-based ratios because customer demand varies very little after the step (only by noise), while orders swing a lot.

*Extend.* From week 40 on the ratios fall to 1.6, 2.1 and 2.2: once the step's swing is over, what is left is mostly the noise we added, which is a little larger in the order series than in demand. The bullwhip ratio depends on the window and on what disturbed the chain within it. Report it with the window and the disturbance, and do not compare ratios measured over different events.

</details>

In [ ]:
# Check: weeks between the peaks of neighbouring stages
pk = summary["peak week"]
print("peak weeks:", pk.to_dict())
# Apply: standard deviation of orders after the step over that of demand
sd = ref.loc[5:].std()
print("ratio of standard deviations, week 5 on:",
      (sd / sd["Customer demand"]).round(1).to_dict())
# Extend: the same ratio from week 40 on, when the swing has died down
late = ref.loc[40:].std()
print("from week 40:", (late / late["Customer demand"]).round(1).to_dict())

## 2. Dynamic hypothesis and boundary

A **dynamic hypothesis** (chapter 2) is a statement of the feedback structure we believe produces the reference mode. Ours is short: *each stage corrects its inventory through a delay with a rule that counts too little of what it has already ordered, and forecasts demand from the orders it receives rather than from what customers buy; each stage's swing then becomes the next stage's demand*.

The rule each stage uses is the **Stock Management** molecule (`stockmgmt`) of chapter 6, after Hines, *Molecules of Structure* [@hines_molecules]; like every molecule in this book it is our own implementation of a structure Hines describes. It orders enough to replace expected demand, to close the inventory gap over an adjustment time, and to close a share $w$ of the supply line gap (the **supply line weight**):

$$
\text{Orders} = \max\!\Big(0,\ \hat D + \frac{S^{*} - S}{T_{\text{adj}}} + w\,\frac{SL^{*} - SL}{T_{\text{adj}}}\Big), \qquad SL^{*} = \hat D \times \lambda ,
$$

with expected demand $\hat D$, inventory $S$ and its target $S^{*}$, supply line $SL$ and its target $SL^{*}$, and the lead time $\lambda$. In the molecule the demand is known exactly and the target inventory is fixed. In the chain, each stage forecasts demand and sets its targets from the forecast. Before building the chain, recall how the molecule behaves on its own when it starts below target.

In [ ]:
sm = lb.molecule("stockmgmt")
print(sm.equations())
for w in (0, 0.25, 1):
    d = sm.run(T=100, supply_line_weight=w)
    print(f"supply line weight {w:<4}: inventory peak "
          f"{d['Inventory'].max():5.1f}, orders {d['Orders'].min():4.1f} "
          f"to {d['Orders'].max():4.1f} units/week")

Starting with 60 units against a target of 100, the molecule overshoots to 110.2 units when it ignores the supply line (weight 0) and to 104.9 at a weight of 0.25, and settles without overshooting at a weight of 1. That is the single-stage mechanism. Three things turn it into a bullwhip in a chain, and they set the **boundary** of our model:

| Element | In the model as | Why |
|---|---|---|
| Inventory, goods in transit, production line | stocks, one set per stage | the material delays the rule must manage |
| Backlog of unfilled orders | a stock per stage | shortages upstream lengthen the lead time downstream |
| Expected demand | a stock per stage (exponential smoothing, chapter 4) | each stage sees only the orders it receives |
| Ordering rule | Stock Management, with the targets set from the forecast | the decision whose weight on the supply line we test |
| Customer demand | exogenous: a step, or weekly noise | the disturbance, not explained by the model |
| Prices, factory capacity, raw materials, workforce | left out | we test ordering policy; chapter 6 covers workforce and the mini-project adds capacity |

Two balancing loops run through every stage: B1, the **inventory loop** (orders raise arrivals, which raise inventory, which cuts orders), and B2, the **supply line loop** (orders raise the supply line, which cuts orders at once). B2 has no delay, B1 has the whole lead time in it. A stage that gives the supply line little weight is relying on the slow loop. Between the stages there is no loop at all, only a chain of influence: the wholesaler's demand is the retailer's orders. That is why the swing can grow and is never corrected at the source.

**Try it 2.**

- *Check.* What desired supply line does the molecule keep at its default settings?
- *Apply.* Start the molecule in balance (inventory at 100) and step demand from 10 to 11 units a week in week 5. Compare the amplification ratio of orders at weights 0.25 and 1, and the range of inventory.
- *Extend.* Make the desired inventory 10 weeks of demand instead of a fixed 100 units. What happens to the amplification ratio at weight 1, and why?

<details><summary>Worked answer</summary>

*Check.* 60 units: 10 units a week of demand times the 6-week acquisition delay.

*Apply.* At weight 0.25, orders peak at 11.68 units a week in week 9.0 (ratio 1.68) and inventory moves between 97.4 and 100.3. At weight 1, orders peak higher, at 12.50 in week 5.0 (ratio 2.50), and inventory stays between 98.2 and 100.0. The peak ratio misleads here: counting the supply line orders the extra pipeline at once (6 more units in transit, over the 4-week adjustment time), then never overshoots. In a single stage, a high peak that is over quickly is not a problem; in a chain it is the next stage's demand, which is why we also look at what follows the peak.

*Extend.* The ratio doubles to 5.00. When the target inventory is 10 weeks of demand, a lasting rise of 1 unit a week raises the target by 10 units, and the rule orders that at once on top of the new demand. Targets that move with the forecast are a source of amplification even for a rule that sees everything: the chain must really build more stock, and the extra orders to do it are a real, not a mistaken, part of the swing.

</details>

In [ ]:
# Check: the supply line the molecule keeps in balance
print("desired supply line:", sm.run(T=0)["Desired supply line"].iloc[0])
# Apply: start in balance (inventory at 100) and step demand from 10 to 11
calm = sm.with_init("Inventory", 100).with_equation(
    "Demand", "10 + STEP(1, 5)")
for w in (0.25, 1):
    d = calm.run(T=100, supply_line_weight=w)
    o = d["Orders"]
    print(f"weight {w}: orders peak {o.max():.2f} units/week in week "
          f"{o.idxmax():.1f}; amplification {(o.max() - 10) / 1:.2f}; "
          f"inventory {d['Inventory'].min():.1f} to "
          f"{d['Inventory'].max():.1f}")
# Extend: a desired inventory that is 10 weeks of demand
grow = calm.with_equation("Desired inventory", "10 * Demand")
o = grow.run(T=100, supply_line_weight=1)["Orders"]
print(f"coverage 10 weeks, weight 1: orders peak {o.max():.2f}, "
      f"amplification {(o.max() - 10) / 1:.2f}")

## 3. The model in code

We build the chain with `lb.Model.from_spec`, as in chapters 6 and 11, from a dictionary of stocks, flows and variables. Rather than write 59 elements by hand, we write small helpers that add one element each, then a function that adds one whole stage, and call it three times. The helpers are those of chapter 11 with one change: they append to a spec that is passed in. `lb.name_key` turns a display name such as `Retailer inventory` into the key used in equations and keyword arguments; Appendix B (@sec-package) documents it with the rest of the package.

The constants shared by all stages come first. Time is in weeks. Customer demand is 100 cases a week and steps up by `Step size` (a fraction, 0.1 for 10%) in week 5. Goods take 2 weeks in transit between stages and the factory takes 3 weeks to make them. Each stage aims to hold 1 week of expected demand (`Inventory coverage`), corrects its inventory over 2 weeks and smooths its demand signal over 2 weeks. `Supply line weight` starts at 0.25, well below 1, as people in beer game experiments tend to set it [@sterman1989]. `Information sharing` is 0 when each stage sees only its own incoming orders and 1 when every stage sees customer demand.

In [ ]:
def new_spec():
    """An empty model: weekly time, run for 100 weeks in steps of 1/8."""
    return {"name": "Three-stage supply chain", "timeUnit": "week",
            "T": 100, "dt": 0.125, "stocks": [], "flows": [], "vars": []}


def stock(spec, name, init, units="cases", floor=True):
    # lb.name_key gives the key a name has in equations and run() keywords
    spec["stocks"].append({"id": lb.name_key(name), "name": name,
                           "init": str(init), "units": units,
                           "nonNegative": floor})


def flow(spec, name, eq, src=None, dst=None, units="cases/week"):
    f = {"id": lb.name_key(name), "name": name, "eq": eq, "units": units}
    if src:                                  # no src or dst: a cloud
        f["from"] = lb.name_key(src)
    if dst:
        f["to"] = lb.name_key(dst)
    spec["flows"].append(f)


def var(spec, name, eq, units):
    spec["vars"].append({"id": lb.name_key(name), "name": name,
                         "eq": str(eq), "units": units})


def shared(spec):
    """Customer demand and the constants that every stage uses."""
    var(spec, "Base demand", 100, "cases/week")
    var(spec, "Step size", 0.1, "fraction")
    var(spec, "Step time", 5, "week")
    var(spec, "Customer demand",
        "Base_demand * (1 + Step_size * STEP(1, Step_time))", "cases/week")
    var(spec, "Transit delay", 2, "week")
    var(spec, "Production delay", 3, "week")
    var(spec, "Backlog clearing time", 1, "week")
    var(spec, "Minimum shipping time", 0.5, "week")
    var(spec, "Inventory coverage", 1, "week")
    var(spec, "Inventory adjustment time", 2, "week")
    var(spec, "Demand smoothing time", 2, "week")
    var(spec, "Supply line weight", 0.25, "dimensionless")
    var(spec, "Information sharing", 0, "dimensionless")

The stage function adds four groups of elements, in the order of the sketch.

1. **Goods.** An inventory, filled from goods in transit (or, at the factory, from the production line). The stage's **supply line** is everything it has ordered and not received: its orders waiting in the supplier's backlog plus the goods in transit. That is the same as "orders not received" in the Low-Visibility Pipeline Correction molecule of chapter 6, which section 5 returns to.
2. **Backlog.** Orders received wait in a backlog. Shipments are what is asked for this week plus a share of the backlog, limited by what the inventory can supply in a minimum shipping time of half a week: the protected outflow of chapter 5, so no inventory can go below zero. A shipment leaves the stage's inventory and enters the downstream stage's goods in transit; a matching flow `orders filled` takes the order out of the backlog.
3. **Forecast.** `Expected demand` is a stock that closes the gap to the demand signal over the smoothing time, which is exponential smoothing (chapter 4). It has no floor, since a forecast can fall as well as rise.
4. **Decision.** The Stock Management rule with the forecast in place of known demand. The orders flow goes into the supplier's backlog; the factory's goes into its own production line, so its "orders" are production starts. `MAX(0, ...)` keeps orders from going negative: goods cannot be sent back.

The model is then the shared constants plus three calls of the function.

In [ ]:
STAGES = ["Retailer", "Wholesaler", "Factory"]      # downstream first


def add_stage(spec, i):
    """Stocks, flows and the ordering rule of stage i of the chain."""
    k = STAGES[i]
    down = STAGES[i - 1] if i > 0 else None   # who orders from stage k
    up = STAGES[i + 1] if i < 2 else None     # whom stage k orders from
    received = f"{down}_orders" if down else "Customer_demand"

    # 1. Goods: the inventory and the pipeline that refills it
    stock(spec, f"{k} inventory", "Inventory_coverage * Base_demand")
    if up:
        stock(spec, f"{k} in transit", "Transit_delay * Base_demand")
        flow(spec, f"{k} arrivals", f"{k}_in_transit / Transit_delay",
             f"{k} in transit", f"{k} inventory")
        var(spec, f"{k} supply line", f"{up}_backlog + {k}_in_transit",
            "cases")
        var(spec, f"{k} lead time", "Transit_delay", "week")
    else:
        stock(spec, f"{k} production line", "Production_delay * Base_demand")
        flow(spec, f"{k} production", f"{k}_production_line"
             " / Production_delay", f"{k} production line", f"{k} inventory")
        var(spec, f"{k} supply line", f"{k}_production_line", "cases")
        var(spec, f"{k} lead time", "Production_delay", "week")

    # 2. Orders received wait in a backlog until goods are shipped
    stock(spec, f"{k} backlog", 0)
    if not down:
        flow(spec, "Customer orders", "Customer_demand", dst=f"{k} backlog")
    flow(spec, f"{k} shipments",
         f"MIN({received} + {k}_backlog / Backlog_clearing_time,"
         f" {k}_inventory / Minimum_shipping_time)",
         f"{k} inventory", f"{down} in transit" if down else None)
    flow(spec, f"{k} orders filled", f"{k}_shipments", f"{k} backlog")

    # 3. The forecast: exponential smoothing of a demand signal
    var(spec, f"{k} demand signal", f"(1 - Information_sharing) * {received}"
        " + Information_sharing * Customer_demand", "cases/week")
    stock(spec, f"{k} expected demand", "Base_demand", "cases/week",
          floor=False)
    flow(spec, f"{k} forecast change", f"({k}_demand_signal"
         f" - {k}_expected_demand) / Demand_smoothing_time",
         dst=f"{k} expected demand", units="cases/week/week")

    # 4. The decision rule: Stock Management with a forecast
    var(spec, f"{k} desired inventory",
        f"Inventory_coverage * {k}_expected_demand", "cases")
    var(spec, f"{k} desired supply line",
        f"{k}_expected_demand * {k}_lead_time", "cases")
    var(spec, f"{k} indicated orders", f"{k}_expected_demand"
        f" + ({k}_desired_inventory - {k}_inventory)"
        " / Inventory_adjustment_time"
        f" + Supply_line_weight * ({k}_desired_supply_line"
        f" - {k}_supply_line) / Inventory_adjustment_time", "cases/week")
    flow(spec, f"{k} orders", f"MAX(0, {k}_indicated_orders)",
         dst=f"{up} backlog" if up else f"{k} production line")

In [ ]:
def chain_spec():
    spec = new_spec()
    shared(spec)
    for i in range(len(STAGES)):
        add_stage(spec, i)
    return spec


chain = lb.Model.from_spec(chain_spec())
print(chain)
print("\n".join(line for line in chain.equations().splitlines()
                if "Wholesaler" in line))

The model has 12 stocks, 16 flows and 31 variables; the list of parameters is what `run()` accepts as keywords. The equations printed are the wholesaler's: compare them line by line with the Stock Management rule above. Now run the base case: customer demand rises 10% in week 5, and every stage uses the rule as found.

In [ ]:
#| fig-alt: "Two panels over 100 weeks after customer demand steps from 100 to 110 cases a week in week 5. Left, orders: retailer orders peak near 119, wholesaler near 135 and factory near 157 cases a week, each later than the one below it, before all settle at 110. Right, inventories: each dips below its target of about 100 cases and then overshoots, the factory's by most."
base = chain.run()
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
base["Customer demand"].plot(ax=a1, label="Customer demand")
for k in STAGES:
    base[f"{k} orders"].plot(ax=a1, label=f"{k} orders")
    base[f"{k} inventory"].plot(ax=a2, label=k)
a1.set(title="Orders placed", xlabel="time (week)",
       ylabel="orders (cases/week)", xlim=(0, 60))
a2.set(title="Inventory", xlabel="time (week)", ylabel="inventory (cases)",
       xlim=(0, 60))
a1.legend(fontsize=8)
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()

Figure: The model with a 10% step in customer demand in week 5 and a supply line weight of 0.25. Left, orders placed by each stage against customer demand; right, each stage's inventory.

The model reproduces the three features of the reference mode without being fitted to it. Orders swing more up the chain, peak later up the chain, and undershoot after the peak; the factory's orders fall to 69.9 cases a week, well below the new demand of 110. Inventories dip first and then overshoot, by most at the factory. To compare runs we put the measures in one table: the amplification ratio, the inventory range, and a cost with the scoring used in the beer distribution game [@sterman1989]: 0.5 per case per week held in inventory and 1.0 per case per week owed in a backlog (cost units, not a currency).

In [ ]:
HOLD, OWE = 0.5, 1.0     # cost per case per week in stock, and owed


def scorecard(d):
    """Amplification and cost for each stage, from one run with a step."""
    dt = d.index[1] - d.index[0]
    jump = d["Customer demand"].iloc[-1] - d["Customer demand"].iloc[0]
    rows = {}
    for k in STAGES:
        o, inv = d[f"{k} orders"], d[f"{k} inventory"]
        rows[k] = {"peak orders": o.max(), "peak week": o.idxmax(),
                   "lowest orders": o.min(),
                   "amplification": (o.max() - o.iloc[0]) / jump,
                   "lowest inventory": inv.min(),
                   "highest inventory": inv.max(),
                   "holding cost": HOLD * inv.sum() * dt,
                   "backlog cost": OWE * d[f"{k} backlog"].sum() * dt}
    return pd.DataFrame(rows).T


card = scorecard(base)
print(card.round(1).to_string())
print("chain cost:", round(card[["holding cost", "backlog cost"]]
                           .sum().sum()))

The amplification ratios are 1.9 at the retailer, 3.5 at the wholesaler and 5.7 at the factory: a 10-case rise at the till becomes a 57-case rise at the factory, which orders 157.3 cases a week in week 10.4. The factory's inventory swings between 58.7 and 173.5 cases around a target that ends at 110. For this small step the cost hardly shows it: holding costs are about 5,400 at every stage, and only the factory runs short, adding a backlog cost of 85. Section 6 shows where the cost of the bullwhip does show: in noisy demand and in larger surges.

**Try it 3.**

- *Check.* By how much does each stage amplify the orders it receives (its ratio over the ratio of the stage below)?
- *Apply.* Run the chain with a 20% step. Do the ratios stay the same?
- *Extend.* Run the chain with inventory adjustment times of 2, 4 and 8 weeks. What happens to amplification, and what would you expect to happen to how long inventories stay away from target?

<details><summary>Worked answer</summary>

*Check.* The retailer amplifies customer demand 1.88 times, the wholesaler the retailer's orders 1.84 times, and the factory the wholesaler's orders 1.65 times. Each stage takes its input and amplifies it again, so the ratios multiply.

*Apply.* With a 20% step the ratios are 2.0, 3.1 and 4.47, and the factory peaks at 189.3 cases a week. The ratios change with the size of the step (the retailer's rises a little, the others fall) because the nonlinear parts of the model act more: shipments limited by inventory, and orders that cannot go below zero. A linear model would give the same ratios for every step size; ours does not, so state the size of the disturbance with every ratio.

*Extend.* Amplification falls as the adjustment time grows: the ratios are [1.88, 3.46, 5.73] at 2 weeks, [1.57, 2.42, 3.59] at 4 and [1.35, 1.81, 2.42] at 8. But the factory's inventory takes longer to settle: it is within 2 cases of its new target for good from week 36.4, 40.6 and 45.1. Slow correction is a cheap way to damp the swing, paid for in time spent away from target.

</details>

In [ ]:
# Check: amplification of each stage relative to the one below it
amp = card["amplification"]
print("stage over stage:", (amp / amp.shift(fill_value=1)).round(2)
      .to_dict())
# Apply: a 20% step instead of 10%
big = scorecard(chain.run(step_size=0.2))
print("20% step:", big["amplification"].round(2).to_dict(),
      "| factory peak", round(big.loc["Factory", "peak orders"], 1))
# Extend: a slower inventory adjustment time
for iat in (2, 4, 8):
    d = chain.run(inventory_adjustment_time=iat)
    s = scorecard(d)
    off = (d["Factory inventory"] - 110).abs() > 2   # more than 2 cases out
    print(f"adjustment time {iat}: amplification",
          s["amplification"].round(2).tolist(), "| factory inventory "
          f"within 2 cases of 110 for good from week {off[off].index[-1]}")

## 4. Tests

Chapter 9 set out the tests that build confidence before a model is used. Three matter most for a chain. **Conservation**: goods must not be created or lost between stages, and each stage's supply line must equal its orders not yet received. **Extreme conditions**: the chain must behave sensibly when demand stops or doubles. **Time step**: the conclusions must not depend on `dt`.

The conservation check uses the same idea as chapter 9: the goods in the chain at any time must equal the starting goods plus everything the factory has started minus everything sold. `cumsum()` adds the net flow up step by step, and `shift(fill_value=0)` moves it down one step, because under Euler's method a flow at time $t$ changes the stocks at $t + dt$.

In [ ]:
full = chain.run(include="all")
dt = chain.dt
GOODS = ["Factory production line", "Factory inventory",
         "Wholesaler in transit", "Wholesaler inventory",
         "Retailer in transit", "Retailer inventory"]
in_chain = full[GOODS].sum(axis=1)
# Euler adds dt x (net flow) at each step, so the books are a running sum
# of what entered (production starts) minus what left (sales), shifted by
# one step because a flow at time t changes the stock at t + dt
books = in_chain.iloc[0] + dt * (full["Factory orders"]
                                 - full["Retailer shipments"]
                                 ).cumsum().shift(fill_value=0)
print(f"goods: largest gap {(in_chain - books).abs().max():.1e} cases")
# The retailer's supply line is its own orders not yet received
open_orders = full["Retailer supply line"].iloc[0] + dt * (
    full["Retailer orders"] - full["Retailer arrivals"]
).cumsum().shift(fill_value=0)
gap = (open_orders - full["Retailer supply line"]).abs().max()
print(f"retailer orders not received: largest gap {gap:.1e} cases")
print("floor at zero acted:", full.attrs["clipped"] or "never")

The goods in the chain match the books to $4.5 \times 10^{-13}$ cases, and the retailer's supply line matches its orders not yet received to $1.1 \times 10^{-13}$ cases: rounding error and nothing else. The floor at zero never acted, so no stock was held up artificially. The second identity matters for the policy that follows: a stage does not need to see inside its supplier to know its supply line. Its own records of orders placed and goods received are enough.

Now the extreme conditions and the time step.

In [ ]:
# 1. No step: a chain started in balance must not move
calm = chain.run(step_size=0, include="stocks")
print("no step: largest move of any stock",
      f"{(calm - calm.iloc[0]).abs().max().max():.1e} cases")
# 2. Customers stop buying in week 5
stop = chain.run(step_size=-1, warn_clips=False)
print("demand stops: lowest stock",
      round(stop[[s["name"] for s in chain.stocks
                  if s["nonNegative"]]].min().min(), 3),
      "| orders in week 100",
      stop.loc[100, [f"{k} orders" for k in STAGES]].round(2).tolist(),
      "| clipped:", stop.attrs["clipped"] or "never")
# 3. Demand doubles in week 5
double = scorecard(chain.run(step_size=1, warn_clips=False))
print("demand doubles: amplification",
      double["amplification"].round(2).tolist(), "| backlog cost",
      double["backlog cost"].round(0).tolist())
# 4. Halve dt: the amplification should not depend on the time step
for step_dt in (0.25, 0.125, 0.0625):
    s = scorecard(chain.run(dt=step_dt))
    print(f"dt {step_dt:<6}: amplification",
          s["amplification"].round(3).tolist())

A chain started in balance with no step does not move at all, which tells us every stock starts at its equilibrium. When customers stop buying in week 5, all orders fall to zero, no stock goes below zero and the floor never acts. When demand doubles, the ratios fall to 1.33, 1.6 and 2.02, and backlogs appear at every stage: costs of 9,460 at the retailer, 5,760 at the wholesaler and 3,096 at the factory. A doubling is far outside normal experience, so we check only that the behaviour is plausible, not that its numbers are right.

Halving `dt` from 0.25 to 0.125 to 0.0625 week moves the factory's amplification from 5.965 to 5.73 to 5.619: the changes shrink by about half with each halving (0.235, then 0.111), as Euler's method should (chapter 9), so the numbers converge. Extrapolating the halvings puts the converged factory ratio near 5.5, so at our `dt` of 0.125 the ratios are about 4% high: good enough to compare policies, and the reason we quote them to two significant figures.

**Try it 4.**

- *Check.* When demand stops, how much inventory does each stage end with, and why is it more than it started with?
- *Apply.* Turn the conservation check into a function `goods_gap(model, **settings)` and an `assert` that passes for a 10% and a 100% step.
- *Extend.* Replace the retailer's shipments with a rule that ships whatever is asked, stock or no stock. Does your test catch it when demand doubles?

<details><summary>Worked answer</summary>

*Check.* The retailer ends with 346.5 cases, the wholesaler 343.8 and the factory 439.9. With no orders, nothing leaves any inventory, but the goods already in transit and on the production line still arrive. Each stage keeps them, since it cannot send them back; that is what the `MAX(0, ...)` in the rule implies.

*Apply.* The function and both asserts are in the cell below; both pass.

*Extend.* It does: the unguarded retailer runs out in week 6.125, the floor at zero holds its inventory up, and the books are out by 501.7 cases. The floor hides the error in the plot (inventory sits at zero, which looks plausible) but not in the books. This is why the protected outflow of the base model, `MIN(..., inventory / minimum shipping time)`, is part of the model and not a detail.

</details>

In [ ]:
# Check: the inventory each stage holds once demand has stopped
print(stop.loc[100, [f"{k} inventory" for k in STAGES]].round(1)
      .to_dict())
# Apply: a test that fails when goods are created or lost
def goods_gap(model, **settings):
    d = model.run(include="all", warn_clips=False, **settings)
    total = d[GOODS].sum(axis=1)
    books = total.iloc[0] + model.dt * (
        d["Factory orders"] - d["Retailer shipments"]
    ).cumsum().shift(fill_value=0)
    return (total - books).abs().max()


assert goods_gap(chain) < 1e-6
assert goods_gap(chain, step_size=1) < 1e-6
print("goods conserved with a 10% and a 100% step")
# Extend: a retailer that ships whatever is asked, stock or no stock
unguarded = chain.with_equation(
    "Retailer shipments", "Customer_demand + Retailer_backlog"
    " / Backlog_clearing_time")
print(f"unguarded, demand doubles: gap "
      f"{goods_gap(unguarded, step_size=1):.1f} cases; floor acted from week",
      unguarded.run(step_size=1, warn_clips=False).attrs["clipped"])

## 5. Explaining the behaviour

The model reproduces the reference mode; now we ask which parts of the structure produce it. Two methods do most of the work: split a decision into its terms and watch each over time, and switch off one part at a time (a **knockout** experiment) to see how much of the behaviour goes with it.

The wholesaler's orders are the sum of three terms: expected demand, the inventory correction and the supply line correction. Plotting them shows what the wholesaler was "thinking" each week.

In [ ]:
#| fig-alt: "Wholesaler orders over 60 weeks split into three parts. Expected demand rises smoothly from 100 to about 116 and back to 110. The inventory correction jumps to about plus 22 cases a week, then turns negative. The supply line correction dips to about minus 5, then turns slightly positive. The total orders peak near 135."
k = "Wholesaler"
iat, w = chain.params["inventory_adjustment_time"], \
    chain.params["supply_line_weight"]
parts = pd.DataFrame({
    "expected demand": full[f"{k} expected demand"],
    "inventory correction": (full[f"{k} desired inventory"]
                             - full[f"{k} inventory"]) / iat,
    "supply line correction": w * (full[f"{k} desired supply line"]
                                   - full[f"{k} supply line"]) / iat})
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
full[f"{k} orders"].plot(ax=a1, label="orders placed")
parts["expected demand"].plot(ax=a1, label="expected demand")
full["Retailer orders"].plot(ax=a1, label="orders received")
parts[["inventory correction", "supply line correction"]].plot(ax=a2)
a2.axhline(0, color="grey", lw=1)
a1.set(title="Wholesaler: orders and forecast", xlabel="time (week)",
       ylabel="cases/week", xlim=(0, 60))
a2.set(title="Wholesaler: the two corrections", xlabel="time (week)",
       ylabel="cases/week", xlim=(0, 60))
a1.legend(fontsize=8)
a2.legend(fontsize=8)
plt.tight_layout()
plt.show()
t = full[f"{k} orders"].idxmax()
print(f"at the peak (week {t}):", parts.loc[t].round(1).to_dict(),
      "| supply line", round(full.loc[t, f"{k} supply line"], 1),
      "against", round(full.loc[t, f"{k} desired supply line"], 1),
      "desired")

Figure: The wholesaler's ordering decision after the 10% step. Left, orders placed, its forecast (expected demand) and the orders it receives from the retailer; right, the inventory correction and the supply line correction (weight 0.25).

At its peak in week 9.5 the wholesaler orders 134.6 cases a week. Of these, 115.2 come from the forecast, which has followed the retailer's inflated orders above the true demand of 110; 22.2 come from the inventory correction, because the wholesaler's inventory has fallen while it waits for goods. The supply line correction is the only term pulling the other way, at minus 2.7: the supply line already holds 252.1 cases against 230.3 wanted, but at a weight of 0.25 most of that excess is ignored. A few weeks later the goods arrive, the inventory correction turns sharply negative, and orders fall below demand. Two mistakes add up: **the forecast is made from a signal that is already amplified**, and **the inventory gap is corrected again and again while the goods to close it are on their way**.

The knockout table runs the step with one part of the structure changed at a time.

In [ ]:
def fixed_targets(model):
    """Targets for inventory and supply line that ignore the forecast."""
    for k in STAGES:
        model = model.with_equation(f"{k} desired inventory",
                                    "Inventory_coverage * Base_demand")
        model = model.with_equation(f"{k} desired supply line",
                                    f"Base_demand * {k}_lead_time")
    return model


experiments = {
    "as found": (chain, {}),
    "count the supply line": (chain, {"supply_line_weight": 1}),
    "share customer demand": (chain, {"information_sharing": 1}),
    "halve the lead times": (chain, {"transit_delay": 1,
                                     "production_delay": 1.5}),
    "fast forecast": (chain, {"demand_smoothing_time": 0.25}),
    "fixed targets": (fixed_targets(chain), {}),
}
knock = pd.DataFrame({name: scorecard(m.run(**p))["amplification"]
                      for name, (m, p) in experiments.items()}).T
print(knock.round(2).to_string())

Reading the factory column: counting the supply line in full cuts the factory's amplification from 5.73 to 4.50; forecasting from customer demand instead of incoming orders cuts it to 3.12; halving the lead times (transit 1 week, production 1.5 weeks) to 4.13. A faster forecast (smoothing over a quarter of a week) makes it worse, 8.23: chasing every movement in incoming orders passes the swing on faster and larger. Fixing the targets for inventory and supply line, so they do not move with the forecast, gives 3.25. That last one is not a policy (the chain really does need more stock in the pipeline when demand is higher), but it tells us how much of the swing comes from targets that move.

So the diagnosis is **ignoring the supply line plus delays**, with the forecast passing the error on. Delays create the gap between ordering and receiving; a low supply line weight turns that gap into over-ordering; and a forecast built from incoming orders carries the over-ordering up the chain as if it were demand.

### Seeing the supply line without seeing the supplier

A manager may reply that the supply line is hard to count because the supplier's processes are invisible. Section 4 showed that it is not: orders not received come from a stage's own records. The **Low-Visibility Pipeline Correction** molecule (`lowVisPipeline`), after Hines, *Molecules of Structure* [@hines_molecules], models exactly this manager. It counts orders not received, infers the delivery delay from them, and weights the pipeline gap by its **awareness of pipeline**. Its *Boom and bust (slow supplier)* preset has a 12-week supplier.

In [ ]:
lv = lb.molecule("lowVisPipeline")
print(lv.preset("Boom and bust (slow supplier)"))
for aware in (0, 0.25, 1):
    d = lv.run(T=150, preset="Boom and bust (slow supplier)",
               awareness_of_pipeline=aware)
    print(f"awareness {aware:<4}: inventory {d['Inventory'].min():5.1f} to "
          f"{d['Inventory'].max():5.1f}, ordering {d['Ordering'].min():4.1f}"
          f" to {d['Ordering'].max():4.1f} units/week, inferred delay "
          f"{d['Calculated delivery delay'].min():4.1f} to "
          f"{d['Calculated delivery delay'].max():4.1f} weeks")

With no awareness of the pipeline, inventory swings between 57.7 and 163.1 units and ordering between 0 and 31.2 units a week against shipments of 10. With full awareness, inventory peaks at only 104.0. Note the third column: the delivery delay the manager infers moves between 7.4 and 20.4 weeks with no awareness, but only between 10.1 and 14.7 with full awareness, around a true delay of 12. The inferred delay is itself distorted by the swing, which matters if a stage uses it to set its supply line target, as the Extend below shows.

**Try it 5.**

- *Check.* At the retailer's peak order, what are its inventory and supply line corrections?
- *Apply.* Fix the targets *and* count the supply line in full. How much amplification is left?
- *Extend.* Let each stage infer its lead time from its own records, as `lowVisPipeline` does (orders not received divided by the rate goods arrive, capped at 30 weeks), and run a doubling of demand at weights 0.25 and 1. Is inferring the lead time a good idea?

<details><summary>Worked answer</summary>

*Check.* At its peak in week 8.125, the retailer's inventory correction is 11.8 cases a week and its supply line correction minus 1.0. The retailer's swing is driven by its inventory gap, as the wholesaler's is.

*Apply.* The ratios fall to 1.14, 1.29 and 1.44. With fixed targets and a full weight, almost all the amplification goes. What is left comes from the forecasts, which lag behind demand: inventories fall a little before the forecasts catch up, and must then be refilled.

*Extend.* No, not without care. With demand doubling, the factory's amplification at weight 0.25 rises from 2.02 with the normal lead time to 4.22 with the inferred one, and at weight 1 from 1.54 to 16.32. When a supplier runs short, deliveries slow, the inferred lead time rises, the desired supply line rises, and the stage orders more, which makes the supplier shorter still: a reinforcing loop. The more weight the stage puts on its supply line, the stronger this loop. Count the supply line, but set its target with a lead time you trust, not one inferred in the middle of a shortage.

</details>

In [ ]:
# Check: the retailer's corrections at its own peak
r = full["Retailer orders"].idxmax()
print(f"retailer peak in week {r}:",
      round((full.loc[r, "Retailer desired inventory"]
             - full.loc[r, "Retailer inventory"]) / iat, 1),
      "inventory correction,",
      round(w * (full.loc[r, "Retailer desired supply line"]
                 - full.loc[r, "Retailer supply line"]) / iat, 1),
      "supply line correction")
# Apply: fixed targets and the supply line counted in full
both = scorecard(fixed_targets(chain).run(supply_line_weight=1))
print("fixed targets, weight 1:", both["amplification"].round(2).tolist())
# Extend: each stage infers its lead time from its own records, as in
# lowVisPipeline: orders not received over the rate goods arrive
inferred = chain
for k, arrive in (("Retailer", "Retailer_arrivals"),
                  ("Wholesaler", "Wholesaler_arrivals"),
                  ("Factory", "Factory_production")):
    inferred = inferred.with_equation(
        f"{k} lead time",
        f"MIN(30, {k}_supply_line / MAX({arrive}, 1e-6))")
for wt in (0.25, 1):
    s = scorecard(inferred.run(supply_line_weight=wt, step_size=1))
    s0 = scorecard(chain.run(supply_line_weight=wt, step_size=1))
    print(f"demand doubles, weight {wt}: factory amplification "
          f"{s0.loc['Factory', 'amplification']:.2f} with the normal "
          f"lead time, {s.loc['Factory', 'amplification']:.2f} inferred")

## 6. Policies and robustness

Three policies follow from the diagnosis, each a change of one or two constants (chapter 11):

| Policy | Change in the model | In practice |
|---|---|---|
| Count the supply line | `supply_line_weight` from 0.25 to 1 | an ordering tool that shows open orders next to stock |
| Share customer demand | `information_sharing` from 0 to 1 | point-of-sale data passed up the chain |
| Halve lead times | `transit_delay` 2 to 1 week, `production_delay` 3 to 1.5 | faster transport and shorter production runs |

We judge them in three scenarios. The first is the 10% step. The second is three years of noisy weekly demand: 100 cases a week plus random noise with a standard deviation of 10 (synthetic, from the chapter's seed), fed into the model with `inputs`, which replaces the `Customer demand` equation with a series. Measured after a 26-week warm-up, it gives the bullwhip ratio by stage, the range of factory orders and the cost per week. The third is a lasting 30% surge in demand, as after a large new contract, where we look at backlog costs. `run_batch` runs every policy in one call; its result is a long table, which `pivot_table` turns back into one wide table per run.

In [ ]:
POLICIES = {
    "As found": {},
    "Count the supply line": {"supply_line_weight": 1},
    "Share customer demand": {"information_sharing": 1},
    "Halve lead times": {"transit_delay": 1, "production_delay": 1.5},
    "All three": {"supply_line_weight": 1, "information_sharing": 1,
                  "transit_delay": 1, "production_delay": 1.5},
}
# One row per policy; constants a policy leaves alone keep their defaults
settings = pd.DataFrame(list(POLICIES.values()),
                        index=list(POLICIES)).fillna(pd.Series(chain.params))

# Scenario 1: the 10% step of the base case
step_cards = {name: scorecard(chain.run(**p)) for name, p in POLICIES.items()}

# Scenario 2: three years of noisy weekly demand (synthetic, seeded)
noise_weeks = np.arange(157)
noisy = pd.Series(100 + rng.normal(0, 10, noise_weeks.size),
                  index=noise_weeks.astype(float))
KEEP = (["Customer demand"]
        + [f"{k} {v}" for k in STAGES
           for v in ("orders", "inventory", "backlog")])
out = chain.run_batch(settings, T=156, include=KEEP,
                      inputs={"Customer demand": noisy})
# out is long (one row per run, time and variable); pivot_table makes it
# wide, with one column per (run, variable) pair, so wide["As found"] is a
# run in the same shape that run() returns
wide = out.pivot_table(index="time", columns=["run", "variable"],
                       values="value", observed=True)


def noise_card(d, start=26):
    """Bullwhip ratios (standard deviation of orders over that of demand)
    and cost per week, measured after a warm-up of `start` weeks."""
    d = d[d.index >= start]
    dt = d.index[1] - d.index[0]
    weeks_run = d.index[-1] - d.index[0]
    sd = d["Customer demand"].std()
    row = {f"{k} ratio": d[f"{k} orders"].std() / sd for k in STAGES}
    row["factory low"] = d["Factory orders"].min()
    row["factory peak"] = d["Factory orders"].max()
    row["holding/week"] = HOLD * sum(d[f"{k} inventory"].sum()
                                     for k in STAGES) * dt / weeks_run
    row["backlog/week"] = OWE * sum(d[f"{k} backlog"].sum()
                                    for k in STAGES) * dt / weeks_run
    row["cost/week"] = row["holding/week"] + row["backlog/week"]
    return row


# Scenario 3: a lasting 30% surge in demand, as after a big new contract
surge = {name: scorecard(chain.run(step_size=0.3, **p))
         for name, p in POLICIES.items()}

table = pd.DataFrame({name: {
    **{f"step: {k}": step_cards[name].loc[k, "amplification"]
       for k in STAGES},
    **noise_card(wide[name]),
    "surge backlog cost": surge[name]["backlog cost"].sum()}
    for name in POLICIES}).T
print(table.round(1).to_string())

Every policy reduces amplification at every stage, and together they nearly remove it. In the noisy scenario the factory's bullwhip ratio is 5.6 as found, 3.5 when the supply line is counted, 2.0 with shared customer demand, 3.3 with halved lead times and 1.2 with all three. The cost shows where it did not in the small step: as found, the chain pays 17.7 a week in backlogs (167.1 a week in all), against 1.7 to 2.4 a week for any single policy and nothing with all three. Holding costs barely move (149 to 150 a week), because each stage's inventory target does not change. The factory's orders tell the operational story: as found, they range from 0.0 to 161.4 cases a week, so the factory must be ready to make 60% more than average demand and also to stop entirely; with all three policies they stay between 76.8 and 124.4.

The surge column adds a warning. Sharing customer demand alone raises backlog costs in the surge, from 2,458 to 3,653. One reading: with shared data the upstream stages forecast the customers' 30% rise, but not the extra goods the stages below must order to fill their larger pipelines and stocks, so they fall behind. Combined with counting the supply line, sharing works well: all three policies together cut the surge's backlog cost to 367.

In [ ]:
#| fig-alt: "Factory orders over weeks 26 to 156 with noisy customer demand. As found, factory orders swing between zero and about 160 cases a week while customer demand stays mostly between 80 and 120. With all three policies, factory orders follow customer demand closely."
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8),
                             gridspec_kw={"width_ratios": [2, 1]})
span = wide.index >= 26
wide.loc[span, ("As found", "Customer demand")].plot(
    ax=a1, label="customer demand")
for name in ("As found", "All three"):
    wide.loc[span, (name, "Factory orders")].plot(
        ax=a1, label=f"factory orders, {name.lower()}")
a1.set(title="Factory orders with noisy demand", xlabel="time (week)",
       ylabel="orders (cases/week)")
a1.legend(fontsize=8)
for name in POLICIES:
    a2.plot(range(3), [table.loc[name, f"{k} ratio"] for k in STAGES],
            marker="o", label=name)
a2.set_xticks(range(3), STAGES)
a2.set(title="Bullwhip ratio by stage", xlabel="stage",
       ylabel="sd of orders / sd of demand")
a2.legend(fontsize=7)
plt.tight_layout()
plt.show()
zero = wide.loc[span, ("As found", "Factory orders")] == 0
starts = zero & ~zero.shift(fill_value=False)       # first step of a spell
print("as found: factory orders nothing in", int(starts.sum()),
      "separate spells,", zero.sum() * chain.dt, "weeks in all")

Figure: Policies in the noisy-demand scenario (synthetic demand, seeded). Left, the factory's orders as found and with all three policies, against customer demand; right, the bullwhip ratio at each stage for each policy.

As found, the factory's orders swing in waves of several weeks and fall to zero twice, for 4.1 weeks in all, although customers buy every week. With all three policies they follow customer demand. The right panel shows the bullwhip as a slope: steep as found, nearly flat with all three.

How sure can we be? The weakest parts of the model are the behavioural constants: nobody knows exactly how fast managers correct their inventory or update their forecasts. We cross three inventory adjustment times (1, 2 and 4 weeks) with three demand smoothing times (1, 2 and 4 weeks), nine scenarios, and run every policy in each, 45 runs in one call. As in chapter 11, the **regret** of a policy in a scenario is how much worse it does than the best policy in that scenario; here we measure it on the factory's bullwhip ratio.

In [ ]:
# Nine behavioural scenarios: how fast managers correct inventory and
# how fast they update their forecast (both in weeks)
grid = [{"inventory_adjustment_time": a, "demand_smoothing_time": s}
        for a in (1, 2, 4) for s in (1, 2, 4)]
rows, labels = [], []
for scen in grid:
    for name, p in POLICIES.items():
        rows.append({**scen, **p})
        labels.append((name, f"adjust {scen['inventory_adjustment_time']},"
                             f" smooth {scen['demand_smoothing_time']}"))
grid_set = pd.DataFrame(rows).fillna(pd.Series(chain.params))
big = chain.run_batch(grid_set, T=156, include=KEEP,
                      inputs={"Customer demand": noisy})
gwide = big.pivot_table(index="time", columns=["run", "variable"],
                        values="value", observed=True)
res = pd.DataFrame([noise_card(gwide[i]) for i in range(len(labels))],
                   index=pd.MultiIndex.from_tuples(
                       labels, names=["policy", "scenario"]))
factory = res["Factory ratio"].unstack("scenario")
print(factory.round(1).to_string())
regret = factory - factory.min()
print(pd.DataFrame({
    "worst factory ratio": factory.max(axis=1),
    "max regret": regret.max(axis=1),
    "worst cost/week": res["cost/week"].unstack("scenario").max(axis=1),
}).round(1).sort_values("max regret").to_string())

The ranking survives. The combination of all three is best in all nine scenarios, its worst ratio 2.4, and its worst cost 149.2 a week. Of the single policies, sharing customer demand has the smallest worst case (a ratio of 5.1 against 13.7 as found) and the smallest maximum regret (2.8). Counting the supply line helps little when managers update their forecasts within a week: with a smoothing time of 1 week and adjustment times of 1, 2 and 4 weeks, it gives factory ratios of 12.3, 6.7 and 3.0 against 13.7, 6.8 and 2.8 as found. A jumpy forecast passes the swing up the chain whatever the ordering rule; shared data stop it (5.1, 2.3 and 1.3). Fast forecasting is the scenario in which a chain most needs shared data.

**Try it 6.**

- *Check.* Of the three single policies, which has the lowest factory ratio in how many of the nine scenarios?
- *Apply.* A manager says counting all of the supply line is unrealistic and offers half (weight 0.5). Run it on the noisy scenario. What does the factory see?
- *Extend.* Draw a second noise series with a different seed and run the five policies again. Does the ranking hold?

<details><summary>Worked answer</summary>

*Check.* Sharing customer demand is best in 8 scenarios, counting the supply line in 1 (slow forecasting with fast inventory correction).

*Apply.* At weight 0.5 the factory's ratio is 4.43, its orders still fall to 0.0 at times and peak at 155.0 cases a week, and the chain pays 5.74 a week in backlogs (154.59 in all). Half the supply line gives part of the benefit, but the factory still has to stop production at times.

*Extend.* With the second series the factory ratios are 5.83 as found, 3.53 counting the supply line, 2.2 sharing customer demand, 3.35 halving lead times and 1.2 with all three: the same order as with the first series. A ranking that holds across noise series and across behavioural scenarios is one we can put in front of a decision maker.

</details>

In [ ]:
# Check: which single policy has the lowest factory ratio, scenario by
# scenario (leaving out the combination)
single = factory.drop(index="All three")
print(single.idxmin().value_counts().to_dict())
# Apply: half the supply line counted (weight 0.5)
half = noise_card(chain.run(T=156, supply_line_weight=0.5,
                            inputs={"Customer demand": noisy}))
print("weight 0.5:", {k: round(float(v), 2) for k, v in half.items()})
# Extend: a second noise series, to see whether the ranking holds
rng2 = np.random.default_rng(SEED + 1)
noisy2 = pd.Series(100 + rng2.normal(0, 10, noise_weeks.size),
                   index=noise_weeks.astype(float))
out2 = chain.run_batch(settings, T=156, include=KEEP,
                       inputs={"Customer demand": noisy2})
wide2 = out2.pivot_table(index="time", columns=["run", "variable"],
                         values="value", observed=True)
print({name: round(noise_card(wide2[name])["Factory ratio"], 2)
       for name in POLICIES})

## Check it

Three habits for any supply chain model: every element has a unit; each pipeline in a new equilibrium holds its flow times its delay (Little's law, chapter 4); and every stage ends ordering what customers buy.

In [ ]:
# 1. Units: every element of the chain has one
print("missing units:", [e["name"] for e in chain.stocks + chain.flows
                         + chain.vars if not e.get("units")])
# 2. Little's law: in the new balance each pipeline holds rate x delay
end = base.index[-1]
print("Wholesaler in transit at week 100:",
      round(full.loc[end, "Wholesaler in transit"], 2), "| arrivals x delay:",
      round(full.loc[end, "Wholesaler arrivals"] * 2, 2))
# 3. Balance: every stage ends with orders equal to customer demand
print("orders in week 100:",
      base.loc[end, [f"{k} orders" for k in STAGES]].round(3).tolist())

Every element has a unit. In week 100 the wholesaler's goods in transit are 220.0 cases, its arrivals of 110 cases a week times the 2-week transit delay; and all three stages order 110.0 cases a week, exactly customer demand. A chain that ended with stages ordering different amounts would be creating or losing goods somewhere.

## Tell the decision maker

- **The swings at the factory are made inside the chain, not by customers.** A 10% rise in shop sales became a 57% rise in factory orders within six weeks, followed by a cut below the new level. With demand that varies from week to week, factory orders varied 5.6 times as much as sales and at times stopped altogether.
- **The cause is how orders are placed.** Each stage chases its stock gap while the goods to fill it are already on the way, and forecasts from the orders it receives, which are already inflated. Long delivery times make both mistakes larger.
- **Three measures fix most of it, and work best together.** Counting open orders in every ordering decision, passing shop sales data up the chain, and halving delivery times together cut the factory's swing to 1.2 times that of sales, removed backlog costs, and kept factory orders between 77 and 124 cases a week instead of 0 to 161.
- **If only one is possible, share sales data, but not alone in a surge.** It was the best single measure in 8 of 9 behavioural scenarios, but in a lasting 30% surge it raised backlog costs unless stages also count their open orders.
- **The data already exist.** Open orders come from each firm's own records; no firm needs to see inside its supplier.

## Going further

- **Other causes of the bullwhip.** Our chain amplifies through ordering rules and delays alone. Ordering in batches, price promotions and rationing (customers inflating orders when a supplier allocates scarce stock) are further causes [@lee1997]. The Extend of Try it 5 is a small version of the last: an inferred lead time that grows in a shortage makes stages order more.
- **Stability without simulation.** Linearise the chain around its equilibrium and compute the eigenvalues of its Jacobian with `numpy.linalg.eigvals`, as suggested in chapter 6. The largest real part tells you how fast a swing dies out, for every combination of weight, adjustment time and delay at once.
- **Discrete orders.** Real orders are placed once a week or once a month, not continuously. Running the model with `dt=1` and a weekly decision is closer to the beer game; compare the ratios with ours and explain the difference (chapter 1 on `dt`).
- **Vendor-managed inventory.** In some chains the supplier manages the retailer's stock directly, which removes one ordering decision. You can model it by giving the wholesaler the retailer's inventory and supply line in its rule.

## Mini-project

Turn the model into the four-stage chain of the beer distribution game by adding a distributor between the wholesaler and the factory (one more call of `add_stage`, after extending `STAGES`). Then give the factory a capacity of 120 cases a week (hint: wrap the factory's orders in `MIN(Production_capacity, ...)` with `with_equation`). With noisy demand, compare the five policies on the bullwhip ratio by stage, the cost per week and the share of weeks the factory runs at capacity. Write a half-page brief for the chain's managers in the style above: what is happening, what to do, what it buys, what it costs, and how sure you are.

## Further reading

- @forrester1961, *Industrial Dynamics*, first modelled the amplification of orders along a production and distribution chain.
- @sterman1989 reports the beer distribution game experiments: how people weight the supply line, and why the bullwhip appears even when customer demand barely changes.
- @sterman2000 treats supply chains, the beer game and stock management at length.
- @lee1997 names the bullwhip effect in operations management and sets out its four classic causes.
- Hines, *Molecules of Structure* [@hines_molecules], describes the stock management and pipeline structures used here.

If you would like to play with the two molecules behind this chapter, the LoopBricks app has them: [Stock Management](https://waelrash1.github.io/loopbricks/#/m/stockmgmt) and [Low-Visibility Pipeline Correction](https://waelrash1.github.io/loopbricks/#/m/lowVisPipeline).